In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets flask-ngrok datasets pyngrok

In [ ]:
!ngrok config add-authtoken 'your auth token here'

In [ ]:
from pyngrok import ngrok
from flask import Flask, request, jsonify,send_file
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig
import torch
from datasets import Dataset
from huggingface_hub import login
import shutil
import os
token='hugging face token'
login(token)

drive.mount('/content/drive')

app=Flask(__name__)

@app.route('/train', methods=['POST'])
def handle_fine_tune():
  try:
    req = request.get_json()
    data_st = req.get('dataset', [])
    params = req.get('params', {})

    epochs = params.get('epochs', 3)
    batch_size = params.get('batch_size', 2)
    max_steps = params.get('max_steps', 10)
    modelx_id = params.get('base_model', 'google/gemma-2b')
    learning_rate = params.get('learning_rate', 2e-4)
    output_directory = "/content/drive/MyDrive/gemma-finetuned"
    dropout = params.get('dropout', 0.05)







    model_id = "google/gemma-2b"

    # load model in 4 bit
    model_id = "google/gemma-2b"
    # load model in 4bit
    model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_4bit=True,
    device_map="auto",
    torch_dtype=torch.float16
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    #lora config
    model = prepare_model_for_kbit_training(model)
    lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], lora_dropout=0.05)
    model = get_peft_model(model, lora_config)



    #tokenise data
    data = data_st

    formatted_data = [
        {"text": f"<|user|>\n{item['input']}\n<|assistant|>\n{item['output']}"} for item in data
    ]
    dataset = Dataset.from_list(formatted_data)
    tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")
    tokenizer.padding_side = "right"
    tokenizer.truncation_side = "left"

    def tokenize(example):
        tokens = tokenizer(
            example["text"],
            truncation=True,
            padding="max_length",
            max_length=256
        )
        tokens["labels"] = tokens["input_ids"].copy()
        return tokens

    tokenized_dataset = dataset.map(tokenize)



    #train the model
    training_args = TrainingArguments(
        output_dir=output_directory,
        per_device_train_batch_size=batch_size,
        num_train_epochs=epochs,
        save_strategy="epoch",
        logging_steps=10,
        fp16=True,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset
    )

    trainer.train()
    print('working till here')
    model = model.merge_and_unload()
    model.save_pretrained("/content/drive/MyDrive/gemma-2b-fullmodel-merged")
    tokenizer.save_pretrained("/content/drive/MyDrive/gemma-2b-fullmodel-merged")
    shutil.make_archive("/content/gemma_model", 'zip', "/content/drive/MyDrive/gemma-2b-fullmodel-merged")

    print('fine tuning finished')
    return jsonify({
    "status": "success",
    "download_url": f"{public_url}/download"
}), 200

  except Exception as e:
     return jsonify({"error": str(e)}), 500


@app.route('/download', methods=['GET'])
def download_model():
    path = "/content/gemma_model.zip"
    if os.path.exists(path):
        return send_file(path, as_attachment=True)
    return jsonify({"error": "File not found"}), 404


public_url = ngrok.connect(5000)
print(f"🌐 Public URL: {public_url}")
app.run()